In [1]:
import requests
import json

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

with open("overpass_radares.txt", "r", encoding="utf-8") as f:
    query = f.read()

r = requests.post(
    OVERPASS_URL,
    data=query.encode("utf-8"),
    headers={
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "User-Agent": "radarwarn-dev/0.1"
    },
    timeout=60
)

print("Status:", r.status_code)
print(r.text[:1000])

r.raise_for_status()

data = r.json()

with open("osm_raw.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Elementos OSM:", len(data.get("elements", [])))

Status: 200
{
  "version": 0.6,
  "generator": "Overpass API 0.7.62.11 87bfad18",
  "osm3s": {
    "timestamp_osm_base": "2026-05-30T17:50:02Z",
    "copyright": "The data included in this document is from www.openstreetmap.org. The data is made available under ODbL."
  },
  "elements": [

{
  "type": "node",
  "id": 21473125,
  "lat": 40.4743622,
  "lon": -3.7433394,
  "tags": {
    "highway": "speed_camera"
  }
},
{
  "type": "node",
  "id": 21682736,
  "lat": 40.4228909,
  "lon": -3.7276999,
  "tags": {
    "direction": "120",
    "highway": "speed_camera",
    "maxspeed": "70"
  }
},
{
  "type": "node",
  "id": 21713934,
  "lat": 40.5248557,
  "lon": -3.6490818,
  "tags": {
    "highway": "speed_camera",
    "maxspeed": "100"
  }
},
{
  "type": "node",
  "id": 25687321,
  "lat": 40.4693425,
  "lon": -3.7128561,
  "tags": {
    "direction": "200",
    "highway": "speed_camera",
    "maxspeed": "50"
  }
},
{
  "type": "node",
  "id": 25687330,
  "lat": 40.4600392,
  "lon": -3.7289478

In [ ]:
import json

INPUT_FILE = "osm_raw.json"
OUTPUT_FILE = "radares_osm_es.json"

MIN_LAT = 27.0
MAX_LAT = 44.5
MIN_LON = -19.0
MAX_LON = 5.0

def inside_spain_bbox(lat, lon):
    return MIN_LAT <= lat <= MAX_LAT and MIN_LON <= lon <= MAX_LON

def get_tipo(tags):
    if tags.get("highway") == "speed_camera":
        return "radar_fijo"

    if tags.get("enforcement") == "maxspeed":
        return "control_velocidad"

    return "radar_posible"

def get_coord_error_m(tags):
    if tags.get("highway") == "speed_camera":
        return 80

    if tags.get("enforcement") == "maxspeed":
        return 120

    return 200

def get_confidence(tags):
    if tags.get("highway") == "speed_camera":
        return 0.90

    if tags.get("enforcement") == "maxspeed":
        return 0.80

    return 0.60

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

radares = []

for e in data.get("elements", []):
    osm_type = e.get("type")
    osm_id = e.get("id")
    tags = e.get("tags", {})

    if "lat" in e and "lon" in e:
        lat = float(e["lat"])
        lon = float(e["lon"])
    elif "center" in e:
        lat = float(e["center"]["lat"])
        lon = float(e["center"]["lon"])
    else:
        continue

    if not inside_spain_bbox(lat, lon):
        continue

    tipo = get_tipo(tags)

    radar = {
        "id": f"osm_{osm_type}_{osm_id}",
        "tipo": tipo,
        "lat": round(lat, 7),
        "lon": round(lon, 7),
        "fuente": "OSM",
        "precision": "alta" if tipo == "radar_fijo" else "media",
        "confidence": get_confidence(tags),
        "coordErrorM": get_coord_error_m(tags),
        "alertDistanceM": 5000,
        "osm": {
            "type": osm_type,
            "id": osm_id
        },
        "tags": tags
    }

    if "maxspeed" in tags:
        radar["maxspeed"] = tags["maxspeed"]

    if "direction" in tags:
        radar["direction"] = tags["direction"]

    if "camera:direction" in tags:
        radar["cameraDirection"] = tags["camera:direction"]

    if "description" in tags:
        radar["description"] = tags["description"]

    radares.append(radar)

vistos = set()
limpios = []

for r in radares:
    key = (
        round(r["lat"], 5),
        round(r["lon"], 5),
        r["tipo"]
    )

    if key not in vistos:
        vistos.add(key)
        limpios.append(r)

limpios.sort(key=lambda r: (r["lat"], r["lon"], r["id"]))

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(limpios, f, ensure_ascii=False, indent=2)

print("Radares limpios:", len(limpios))
print("Guardado en:", OUTPUT_FILE)

FileNotFoundError: [Errno 2] No such file or directory: 'osm_raw.json'

In [ ]:
import requests
import xml.etree.ElementTree as ET
import json

URL = "https://infocar.dgt.es/datex2/dgt/PredefinedLocationsPublication/radares/content.xml"

xml = requests.get(URL, timeout=60).content
root = ET.fromstring(xml)

def clean_tag(tag):
    return tag.split("}")[-1]

radares = []

for elem in root.iter():
    tag = clean_tag(elem.tag).lower()

    if tag in ["pointcoordinates", "coordinatesforpoint"]:
        lat = None
        lon = None

        for child in elem.iter():
            name = clean_tag(child.tag).lower()
            text = child.text.strip() if child.text else None

            if name == "latitude" and text:
                lat = float(text)
            elif name == "longitude" and text:
                lon = float(text)

        if lat is not None and lon is not None:
            radares.append({
                "id": f"dgt_{len(radares) + 1}",
                "tipo": "radar_fijo",
                "lat": lat,
                "lon": lon,
                "fuente": "DGT",
                "precision": "media",
                "confidence": 0.65,
                "tags": {}
            })

# deduplicar
vistos = set()
limpios = []

for r in radares:
    key = (round(r["lat"], 6), round(r["lon"], 6))
    if key not in vistos:
        vistos.add(key)
        limpios.append(r)

with open("radares_dgt_es.json", "w", encoding="utf-8") as f:
    json.dump(limpios, f, ensure_ascii=False, indent=2)

print("Radares DGT:", len(limpios))

In [ ]:
import json
from math import radians, sin, cos, sqrt, atan2

def distancia_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1 = radians(lat1)
    p2 = radians(lat2)
    dp = radians(lat2 - lat1)
    dl = radians(lon2 - lon1)

    a = sin(dp / 2) ** 2 + cos(p1) * cos(p2) * sin(dl / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

with open("radares_osm_es.json", "r", encoding="utf-8") as f:
    osm = json.load(f)

with open("radares_dgt_es.json", "r", encoding="utf-8") as f:
    dgt = json.load(f)

final = []

# Primero metemos OSM porque es más preciso
for r in osm:
    final.append(r)

# Luego añadimos DGT solo si no hay uno OSM cerca
for d in dgt:
    mejor = None

    for o in osm:
        dist = distancia_m(d["lat"], d["lon"], o["lat"], o["lon"])

        if mejor is None or dist < mejor:
            mejor = dist

    if mejor is not None and mejor <= 300:
        continue

    d["precision"] = "media"
    d["confidence"] = 0.6
    d["nota"] = "No se encontró radar OSM cercano; coordenada oficial aproximada"
    final.append(d)

with open("radares_espana.json", "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=2)

print("Total final:", len(final))

In [ ]:
import json

with open("radares_espana.json", "r", encoding="utf-8") as f:
    radares = json.load(f)

dudosos = [
    r for r in radares
    if r.get("confidence", 0) < 0.75
]

with open("radares_dudosos.json", "w", encoding="utf-8") as f:
    json.dump(dudosos, f, ensure_ascii=False, indent=2)

print("Dudosos:", len(dudosos))